In [1]:
!pip install pyspark

In [2]:
import zipfile

# Extract Q1
with zipfile.ZipFile('Q1.zip', 'r') as zip_ref:
    zip_ref.extractall()



In [3]:
import os
print(os.listdir())

['.config', 'Q1.zip', '__MACOSX', 'Q1', 'sample_data']


In [4]:
!pip install pyspark

In [5]:
import random
import time
from pyspark.mllib.linalg import Vectors

In [ ]:
from pyspark.mllib.linalg import Vectors

def readVectorsSeq(filename):
    P = []
    with open(filename, 'r') as f:
        for line in f:
            values = list(map(float, line.strip().split(',')))
            P.append(Vectors.dense(values)) 
    return P

In [7]:
import os
print(os.listdir("Q1"))

['spambase.names', 'spambase.data', 'spambase.DOCUMENTATION']


In [8]:
P = readVectorsSeq("Q1/spambase.data")  # adjust if name different
print(len(P))

4601


In [14]:
def distance(a, b):
    return sum((a[i] - b[i]) ** 2 for i in range(len(a)))

In [15]:
def kcenter(P, k):
    C = [random.choice(P)]

    while len(C) < k:
        max_dist = -1
        next_center = None

        for p in P:
            min_dist = float('inf')
            for c in C:
                dist = distance(p, c)
                if dist < min_dist:
                    min_dist = dist

            if min_dist > max_dist:
                max_dist = min_dist
                next_center = p

        C.append(next_center)

    return C

In [16]:
def kmeansPP(P, k):
    C = [random.choice(P)]

    while len(C) < k:
        distances = []

        for p in P:
            min_dist = min(distance(p, c) for c in C)
            distances.append(min_dist)

        total = sum(distances)
        probs = [d / total for d in distances]

        r = random.random()
        cumulative = 0

        for i, p in enumerate(P):
            cumulative += probs[i]
            if r <= cumulative:
                C.append(p)
                break

    return C

In [17]:
def kmeansObj(P, C):
    total = 0

    for p in P:
        min_dist = min(distance(p, c) for c in C)
        total += min_dist

    return total / len(P)

In [18]:
k = 10
k1 = 20

# 1. k-center
start = time.time()
C1 = kcenter(P, k)
end = time.time()
print("k-center time:", end - start)

# 2. kmeans++
C2 = kmeansPP(P, k)
obj1 = kmeansObj(P, C2)
print("kmeans++ objective:", obj1)

# 3. Hybrid approach
X = kcenter(P, k1)
C3 = kmeansPP(X, k)
obj2 = kmeansObj(P, C3)
print("Hybrid objective:", obj2)

k-center time: 7.386229038238525
kmeans++ objective: 28321.348255062338
Hybrid objective: 1827659.1740987787
